In [ ]:
"""
evaluate_cg_bench.ipynb
Evaluates the ten causal-discovery methods implemented in
../causal_discovery.py against real/externally-sourced benchmark ground
truth (as opposed to evaluate_cg.ipynb, which uses self-authored
synthetic data).

  IID methods         : PC, FCI, CD-NOD, LiNGAM, DAG-GNN, GES
                        -> scored against Sachs, a real protein-signaling
                        dataset (Sachs et al. 2005), 11 variables,
                        18 ground-truth edges. See
                        datasets/dataset_generation.txt for the full
                        citation and provenance.
  Time-series methods : PCMCI, PCMCI+, TCDF, LPCMCI
                        -> scored against a nonlinear, confounded
                        benchmark ported from TimeGraph (Ferdous,
                        Hossain, Gani -- KDD 2025), 8 variables, lag up
                        to 3, 9 ground-truth edges. A latent confounder U
                        affects X1 and X8. U IS present as a CSV column
                        but is deliberately EXCLUDED from the columns
                        passed to run_*() below, so the confounding stays
                        genuinely hidden from the methods being tested --
                        a fabricated direct X1<->X8 edge from a non-PAG
                        method is an EXPECTED artifact of this design,
                        not a bug. See datasets/dataset_generation.txt
                        for full details.

Each method is called through its own production run_*() function from
causal_discovery.py -- none of the causal-discovery logic is reimplemented
here. Time-series predictions are scored on the lag-collapsed summary
graph. Reported metrics: SHD (Structural Hamming Distance), F1, FDR
(False Discovery Rate), precision, recall.
"""
import sys, os, json, time
import numpy as np
import pandas as pd

# causal_discovery.py lives one directory above this notebook
sys.path.insert(0, os.path.abspath(".."))

from causal_discovery import (
    run_pc, run_fci, run_cdnod, run_lingam, run_dag_gnn, run_ges,
    run_pcmci, run_pcmci_plus, run_tcdf, run_lpcmci,
    numeric_columns,
)

DATASETS_DIR = "datasets/benchmark"

# Human-readable labels for each method key, matching the display names
# used in the Streamlit app (frontend_light_cg.py, Frontend/Code/)
METHOD_DISPLAY = {
    'pc':'PC','fci':'FCI','cdnod':'CD-NOD','lingam':'LiNGAM',
    'daggnn':'DAG-GNN','ges':'GES','pcmci':'PCMCI','pcmci_plus':'PCMCI+',
    'tcdf':'TCDF','lpcmci':'LPCMCI',
}

# One row per method is appended here as each evaluation cell below runs
results = []

# Raw predicted edge lists, keyed by method display name -- kept alongside
# the aggregate metrics in `results` so a later cell can draw true-vs-
# predicted graph comparisons without re-running every method.
predicted_edges = {}

print("Imported run_* from ../causal_discovery.py")


# Load datasets + ground truth

In [ ]:
def load_dataset_and_truth(name):
    """Loads a prepared dataset and its ground-truth edges from
    DATASETS_DIR. Expects two files: '{name}.csv' (the data) and
    '{name}_truth.json' (a list of {'cause', 'effect', ...} dicts)."""
    df = pd.read_csv(f"{DATASETS_DIR}/{name}.csv")
    with open(f"{DATASETS_DIR}/{name}_truth.json") as f:
        true_edges = json.load(f)
    return df, true_edges

# IID methods are scored against Sachs; time-series methods against the
# nonlinear, confounded benchmark (see the header cell above for the full
# rationale, and datasets/dataset_generation.txt for generation details).
iid_df, iid_truth = load_dataset_and_truth("sachs")
ts_df,  ts_truth  = load_dataset_and_truth("nonlinear_confounded")

# These are prepared benchmark files where every relevant CSV column IS a
# variable, so there is no KG-node-to-column mapping step to run here
# (unlike the live app, which first maps LLM-derived KG nodes to dataset columns).
iid_cols = numeric_columns(iid_df)

# ts_df additionally contains 'U' (the latent confounder) and 'time'
# (the row index), both numeric -- numeric_columns() alone would wrongly
# include them. 'U' must stay excluded so the confounding remains hidden
# from the discovery methods; 'time' is not a variable at all.
ts_cols = [c for c in numeric_columns(ts_df) if c not in ("U", "time")]

print(f"IID : {iid_df.shape}, cols={iid_cols}, {len(iid_truth)} true edges")
print(f"TS  : {ts_df.shape}, cols={ts_cols}, {len(ts_truth)} true edges")


# Scoring harness (the verified metrics)

In [ ]:
def compute_metrics(pred_edges, true_edges):
    """
    Computes two families of accuracy metrics between a method's predicted
    edges and the ground-truth edges. Self-loops are dropped and lags are
    collapsed (time-series predictions are scored on the summary graph --
    edge presence/direction only, ignoring which lag it was found at).

    DIRECTED metrics (the headline numbers for PC, LiNGAM, GES, CD-NOD,
    DAG-GNN, and all time-series methods): require an exact cause->effect
    match, so direction matters.
      SHD (Structural Hamming Distance) is counted per unordered variable
      pair -- any disagreement on a pair (missing, extra, or reversed)
      costs exactly 1, so a reversed edge is not double-counted.

    SKELETON metrics (the headline numbers for the PAG methods FCI and
    LPCMCI, whose bidirected/circle marks are a deliberate "cannot orient
    this edge" statement, not a mistake): direction is ignored, only
    whether an edge exists between two variables. skel_SHD/skel_F1 are
    the best-case (undirected) bound on the directed metrics above.
    """
    def pairmap(edges):
        """Groups edges by the unordered variable pair they connect, so a
        pair can be compared regardless of which direction it was
        predicted in."""
        m = {}
        for e in edges:
            c, ef = e['cause'], e['effect']
            if c == ef:                        # self-loops are not real edges
                continue
            m.setdefault(frozenset((c, ef)), set()).add((c, ef))
        return m

    # Directed SHD: for every pair seen on either side, a mismatch
    # (missing, extra, or wrong direction) costs exactly 1.
    pm, tm = pairmap(pred_edges), pairmap(true_edges)
    shd = sum(1 for pair in (set(pm) | set(tm))
              if pm.get(pair, set()) != tm.get(pair, set()))

    # Directed precision / recall / F1 / FDR: exact (cause, effect) match required.
    pred = {(e['cause'], e['effect']) for e in pred_edges if e['cause'] != e['effect']}
    true = {(e['cause'], e['effect']) for e in true_edges if e['cause'] != e['effect']}
    tp, fp, fn = len(pred & true), len(pred - true), len(true - pred)
    precision = tp/(tp+fp) if (tp+fp) else 0.0
    recall    = tp/(tp+fn) if (tp+fn) else 0.0
    f1  = 2*precision*recall/(precision+recall) if (precision+recall) else 0.0
    fdr = fp/(tp+fp) if (tp+fp) else 0.0

    # Skeleton (undirected) precision / recall / F1 / FDR: direction ignored.
    pred_skel = {frozenset(e) for e in pred}
    true_skel = {frozenset(e) for e in true}
    s_tp = len(pred_skel & true_skel)
    s_fp = len(pred_skel - true_skel)
    s_fn = len(true_skel - pred_skel)
    s_prec = s_tp/(s_tp+s_fp) if (s_tp+s_fp) else 0.0
    s_rec  = s_tp/(s_tp+s_fn) if (s_tp+s_fn) else 0.0
    skel_f1  = 2*s_prec*s_rec/(s_prec+s_rec) if (s_prec+s_rec) else 0.0
    skel_fdr = s_fp/(s_tp+s_fp) if (s_tp+s_fp) else 0.0
    skel_shd = len(pred_skel ^ true_skel)      # missing + extra adjacencies

    return dict(
        # Directed metrics
        SHD=shd, F1=round(f1,3), FDR=round(fdr,3),
        precision=round(precision,3), recall=round(recall,3),
        TP=tp, FP=fp, FN=fn,
        # Skeleton (undirected) metrics
        skel_SHD=skel_shd, skel_F1=round(skel_f1,3),
        skel_FDR=round(skel_fdr,3), skel_precision=round(s_prec,3),
        skel_recall=round(s_rec,3),
        # Raw counts
        n_pred=len(pred), n_true=len(true),
    )


def evaluate(method_key, run_fn, df, cols, true_edges, family, **kwargs):
    """
    Runs one causal-discovery method, times it, scores its output against
    the ground truth, and records one result row.

    method_key : short key used to look up the display name (e.g. 'pc')
    run_fn     : the actual run_*() function from causal_discovery.py
    df, cols   : the dataset and the column names to run the method on
    true_edges : ground-truth edges to score against
    family     : 'IID' or 'TimeSeries', shown in the results table
    **kwargs   : extra arguments forwarded to run_fn (e.g. alpha, tau_max)

    A method that raises an exception is recorded with an 'error' field
    instead of stopping the notebook, so one broken method never blocks
    the rest. Re-running this for the same method replaces its previous
    row instead of duplicating it.
    """
    name = METHOD_DISPLAY[method_key]
    print(f"\n{'─'*55}\n▶ {name}  ({family})")
    t0 = time.time()
    try:
        pred_edges = run_fn(df, cols, **kwargs)
        predicted_edges[name] = pred_edges
        elapsed = time.time() - t0
        m = compute_metrics(pred_edges, true_edges)
        row = {'method': name, 'family': family, 'time_s': round(elapsed, 1), **m}
        print(f"  {m['n_pred']} edges | SHD={m['SHD']} F1={m['F1']} FDR={m['FDR']} "
              f"| skel_SHD={m['skel_SHD']} skel_F1={m['skel_F1']}  ({elapsed:.1f}s)")
    except Exception as e:
        row = {'method': name, 'family': family, 'time_s': None,
               'SHD': None, 'F1': None, 'FDR': None, 'error': str(e)}
        print(f"  ERROR: {e}")
    # Replace any previous row for this method before appending the new one
    results[:] = [r for r in results if r.get('method') != name]
    results.append(row)
    return row


# Cell 3b — notebook-only PAG eval wrappers (FCI, LPCMCI).

In [ ]:
# Notebook-only evaluation wrappers for the two PAG (Partial Ancestral
# Graph) methods, FCI and LPCMCI. Each wrapper mirrors its production
# run_*() function in causal_discovery.py exactly -- same underlying
# call, same statistical test, same endpoint-mark classification -- but
# additionally returns the full skeleton and a count of bidirected marks,
# both of which the production functions discard. No decision differs
# from what the live app does: the directed edges returned here are
# identical to what run_fci()/run_lpcmci() would produce. This wrapper
# only exposes extra structure so the skeleton-based metrics below can be
# computed.

def run_fci_eval(df, columns, alpha=0.05):
    """FCI (Fast Causal Inference) via causal-learn. Returns
    (directed_edges, skeleton_edges, bidirected_mark_count)."""
    from causallearn.search.ConstraintBased.FCI import fci as cl_fci
    from causallearn.graph.Endpoint import Endpoint
    data = df[columns].dropna().to_numpy()
    _, fci_edges = cl_fci(data, independence_test_method="fisherz", alpha=alpha,
                          show_progress=False, node_names=columns)
    directed, skeleton, bidirected = [], [], 0
    for e in fci_edges:
        n1, n2 = e.get_node1().get_name(), e.get_node2().get_name()
        ep1, ep2 = e.get_endpoint1(), e.get_endpoint2()
        skeleton.append({'cause': n1, 'effect': n2})  # any endpoint mark counts as an adjacency
        # TAIL -> ARROW is a fully resolved direction: n1 causes n2.
        if ep1 == Endpoint.TAIL and ep2 == Endpoint.ARROW:
            directed.append({'cause': n1, 'effect': n2, 'label': 'FCI_CAUSES', 'confidence': 1.0})
        elif ep1 == Endpoint.ARROW and ep2 == Endpoint.TAIL:
            directed.append({'cause': n2, 'effect': n1, 'label': 'FCI_CAUSES', 'confidence': 1.0})
        # ARROW <-> ARROW means "these two share an unmeasured common
        # cause" -- FCI's way of flagging latent confounding instead of
        # guessing a direction. Counted separately, never turned into a
        # directed edge.
        elif ep1 == Endpoint.ARROW and ep2 == Endpoint.ARROW:
            bidirected += 1
    return directed, skeleton, bidirected


def run_lpcmci_eval(df, columns, tau_min=0, tau_max=2, alpha=0.05):
    """LPCMCI (latent-confounder-aware PCMCI) via tigramite. Returns
    (directed_edges, skeleton_edges, bidirected_mark_count)."""
    import tigramite.data_processing as pp
    from tigramite.lpcmci import LPCMCI
    from tigramite.independence_tests.parcorr import ParCorr
    from sklearn.preprocessing import StandardScaler
    data = df[columns].dropna()
    data_scaled = StandardScaler().fit_transform(data.values)
    tig_df = pp.DataFrame(data_scaled, var_names=columns)
    lpcmci = LPCMCI(dataframe=tig_df, cond_ind_test=ParCorr(significance="analytic"), verbosity=0)
    res = lpcmci.run_lpcmci(tau_min=tau_min, tau_max=tau_max, pc_alpha=alpha)
    graph = res["graph"]
    n = len(columns)
    directed, skeleton_pairs, bidirected = {}, set(), 0
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            for tau in range(0, tau_max + 1):
                mark = graph[i, j, tau]
                if not mark:
                    continue
                skeleton_pairs.add(frozenset((columns[i], columns[j])))  # any mark counts as an adjacency
                if mark == "-->":
                    directed[(columns[i], columns[j])] = True
                elif mark == "<--":
                    directed[(columns[j], columns[i])] = True
                # '<->' marks a latent common cause, same meaning as
                # FCI's ARROW<->ARROW above -- kept as a count, never
                # turned into a directed edge.
                elif mark == "<->":
                    bidirected += 1
    directed_edges = [{'cause': c, 'effect': e, 'label': 'LPCMCI', 'confidence': 1.0}
                      for (c, e) in directed]
    skeleton = [{'cause': tuple(p)[0], 'effect': tuple(p)[1]} for p in skeleton_pairs]
    return directed_edges, skeleton, bidirected


def evaluate_pag(method_key, eval_fn, df, cols, true_edges, family, **kwargs):
    """
    Scores a PAG method using its wrapper (run_fci_eval / run_lpcmci_eval
    above). Skeleton metrics are reported as the headline SHD/F1 (the
    fair comparison for a method that deliberately leaves some edges
    unoriented); directed metrics are kept in the same row for
    transparency alongside them. Same error handling and idempotent-row
    behavior as evaluate() above.
    """
    name = METHOD_DISPLAY[method_key]
    print(f"\n{'─'*55}\n▶ {name}  ({family})  [PAG]")
    t0 = time.time()
    try:
        directed, skeleton, bidirected = eval_fn(df, cols, **kwargs)
        predicted_edges[name] = directed
        elapsed = time.time() - t0
        m_dir  = compute_metrics(directed, true_edges)
        m_skel = compute_metrics(skeleton, true_edges)
        row = {
            'method': name, 'family': family, 'time_s': round(elapsed, 1),
            # Headline numbers use the skeleton metrics (the fair comparison for a PAG method)
            'SHD': m_skel['skel_SHD'], 'F1': m_skel['skel_F1'], 'FDR': m_skel['skel_FDR'],
            'precision': m_skel['skel_precision'], 'recall': m_skel['skel_recall'],
            'skel_SHD': m_skel['skel_SHD'], 'skel_F1': m_skel['skel_F1'],
            # Directed metrics kept alongside for transparency, not as the headline
            'directed_SHD': m_dir['SHD'], 'directed_F1': m_dir['F1'],
            'bidirected_marks': bidirected,
            'n_pred': len(directed), 'n_true': m_skel['n_true'],
        }
        print(f"  skeleton: {len(skeleton)} edges | skel_SHD={m_skel['skel_SHD']} "
              f"skel_F1={m_skel['skel_F1']}")
        print(f"  directed: {len(directed)} edges | dir_SHD={m_dir['SHD']} "
              f"dir_F1={m_dir['F1']} | bidirected marks={bidirected}  ({elapsed:.1f}s)")
    except Exception as e:
        row = {'method': name, 'family': family, 'time_s': None,
               'SHD': None, 'F1': None, 'FDR': None, 'error': str(e)}
        print(f"  ERROR: {e}")
    results[:] = [r for r in results if r.get('method') != name]
    results.append(row)
    return row


# IID methods, one per cell (run on iid_df, scored vs the DAG)

In [ ]:
# PC -- CPDAG output: edges it cannot orient are left undirected and
# dropped here (not guessed), so recall may be lower by design. Sachs is
# real biological data, so expect meaningfully lower scores here than on
# a clean synthetic benchmark -- a known, published property of Sachs,
# not a bug.
evaluate('pc', run_pc, iid_df, iid_cols, iid_truth, 'IID', alpha=0.05)


In [ ]:
# LiNGAM -- returns a fully directed DAG. Its identifiability relies on
# non-Gaussian noise; Sachs is real data with no guaranteed noise
# distribution, but LiNGAM is nonetheless a standard baseline applied to
# this dataset in the causal-discovery literature.
evaluate('lingam', run_lingam, iid_df, iid_cols, iid_truth, 'IID')


In [ ]:
# GES -- score-based method (maximizes BIC via greedy search); CPDAG
# output like PC, but arrived at through a different search strategy.
evaluate('ges', run_ges, iid_df, iid_cols, iid_truth, 'IID')


In [ ]:
# CD-NOD -- CPDAG output like PC, additionally uses a row-order context
# index to flag variables whose causal mechanism may be nonstationary.
evaluate('cdnod', run_cdnod, iid_df, iid_cols, iid_truth, 'IID', alpha=0.05)


In [ ]:
# DAG-GNN -- returns a fully directed DAG via continuous optimization
# (a small neural network); slower than the constraint-based methods above.
evaluate('daggnn', run_dag_gnn, iid_df, iid_cols, iid_truth, 'IID')


In [ ]:
# FCI -- PAG method: skeleton metrics are the headline score here (see
# evaluate_pag's docstring above), with directed metrics kept alongside.
evaluate_pag('fci', run_fci_eval, iid_df, iid_cols, iid_truth, 'IID', alpha=0.05)


# Time-series methods, one per cell (run on ts_df, summary-graph scored)

In [ ]:
# PCMCI -- lagged-only method. This benchmark is nonlinear and has a
# hidden confounder (U, excluded above) affecting X1 and X8 -- lower
# scores, and/or a fabricated direct X1<->X8 edge, are expected here (see
# the header cell), not a sign of a bug.
evaluate('pcmci', run_pcmci, ts_df, ts_cols, ts_truth, 'TimeSeries', tau_max=5, alpha=0.05)


In [ ]:
# PCMCI+ -- detects lagged AND contemporaneous (same-timestep) links,
# unlike plain PCMCI above, which is lagged-only. Same confounder caveat
# as PCMCI applies here.
evaluate('pcmci_plus', run_pcmci_plus, ts_df, ts_cols, ts_truth, 'TimeSeries', tau_max=5, alpha=0.05)


In [ ]:
# TCDF -- convolutional neural network + attention; slower than the
# other time-series methods, since it trains one small network per
# target variable. Same confounder caveat as PCMCI applies here.
evaluate('tcdf', run_tcdf, ts_df, ts_cols, ts_truth, 'TimeSeries', tau_max=5)


In [ ]:
# LPCMCI -- PAG method for time series, the lagged counterpart of FCI:
# skeleton metrics are the headline score here, same convention as FCI
# above. This is the one method here designed to correctly flag the
# hidden X1/X8 confounder as a bidirected mark ("shared latent cause")
# instead of fabricating a direction for it, unlike PCMCI/PCMCI+/TCDF above.
evaluate_pag('lpcmci', run_lpcmci_eval, ts_df, ts_cols, ts_truth, 'TimeSeries',
             tau_min=0, tau_max=2, alpha=0.05)


In [ ]:
summary = pd.DataFrame(results)

# Order rows to match the method list order used in the live app: IID
# methods first, then time-series methods.
order = ['PC','FCI','CD-NOD','LiNGAM','DAG-GNN','GES',
         'PCMCI','PCMCI+','TCDF','LPCMCI']
summary['__o'] = summary['method'].map({m:i for i,m in enumerate(order)})
summary = summary.sort_values('__o').drop(columns='__o').reset_index(drop=True)

# Label each row by which metric type is its headline: PAG methods
# (FCI, LPCMCI) report skeleton-based numbers; every other method
# reports directed numbers (see compute_metrics/evaluate_pag above).
PAG_METHODS = {'FCI', 'LPCMCI'}
summary['metric_type'] = summary['method'].apply(
    lambda m: 'skeleton' if m in PAG_METHODS else 'directed')

cols_show = ['method','family','SHD','F1','FDR','precision','recall',
             'metric_type','n_pred','n_true','time_s']
# Also show directed_SHD/directed_F1/bidirected_marks when present
# (only PAG rows have these), for transparency alongside the headline numbers.
for extra in ['directed_SHD','directed_F1','bidirected_marks']:
    if extra in summary.columns:
        cols_show.append(extra)
cols_show = [c for c in cols_show if c in summary.columns]

display(summary[cols_show])
summary.to_csv("evaluation_results_bench.csv", index=False)
print("Saved -> evaluation_results_bench.csv")


# Metric comparison plot (SHD / F1 / FDR)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

# Colors distinguish the two dataset families; hatching flags the two PAG
# methods (FCI, LPCMCI), whose SHD/F1/FDR here are skeleton-based headline
# metrics, not directed ones -- see evaluate_pag()'s docstring above for why.
FAMILY_COLORS = {'IID': '#4C72B0', 'TimeSeries': '#DD8452'}
METRIC_DIRECTION = {
    'SHD': 'lower is better', 'FDR': 'lower is better',
    'F1': 'higher is better', 'precision': 'higher is better',
    'recall': 'higher is better',
}

def plot_metric_bars(summary_df, metrics=('SHD', 'F1', 'FDR'), save_path='evaluation_plot_bench.png'):
    """
    Grouped bar chart, one panel per metric -- the standard comparison
    style used in causal-discovery papers (e.g. NOTEARS, DAG-GNN) for
    reporting SHD/F1/FDR across methods. SHD and FDR get their own panels
    from F1 since they are on different scales/directions (SHD is
    unbounded and lower-is-better; F1 is bounded 0-1 and higher-is-better)
    -- plotting them on one shared axis would be misleading.
    """
    methods = summary_df['method'].tolist()
    x = np.arange(len(methods))

    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 5))
    if len(metrics) == 1:
        axes = [axes]

    for ax, metric in zip(axes, metrics):
        values = summary_df[metric].astype(float).to_numpy()
        colors = [FAMILY_COLORS[fam] for fam in summary_df['family']]
        bars = ax.bar(x, values, color=colors)
        for bar, method in zip(bars, methods):
            if method in PAG_METHODS:
                bar.set_hatch('//')

        ax.set_xticks(x)
        ax.set_xticklabels(methods, rotation=45, ha='right')
        ax.set_title(f"{metric}  ({METRIC_DIRECTION.get(metric, '')})")
        ax.set_ylabel(metric)
        ax.grid(axis='y', linestyle=':', alpha=0.5)

    legend_handles = [Patch(facecolor=c, label=f) for f, c in FAMILY_COLORS.items()]
    legend_handles.append(Patch(facecolor='white', edgecolor='black', hatch='//',
                                 label='PAG method (skeleton-based score)'))
    fig.legend(handles=legend_handles, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.12))
    fig.suptitle('Causal Discovery Method Comparison', y=1.2, fontsize=14, fontweight='bold')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved -> {save_path}")

plot_metric_bars(summary)


# True vs. predicted causal graphs (all methods, one figure)


In [ ]:
import networkx as nx
from matplotlib.lines import Line2D

def _clean_edges(edges):
    """Directed (cause, effect) pairs, self-loops dropped -- matches
    compute_metrics()'s own scoring convention, so the picture never shows
    something the metrics didn't actually count."""
    return {(e['cause'], e['effect']) for e in edges if e['cause'] != e['effect']}


def _layered_layout(true_edges, columns):
    """
    Clean hierarchical node layout built from the TRUE graph's topological
    generations -- no graphviz dependency (pygraphviz/pydot are importable
    in this environment, but the underlying `dot` binary is not installed,
    so graphviz_layout() fails at runtime; verified directly, not assumed).
    Falls back to a fixed-seed spring layout if the true graph is not
    acyclic (shouldn't happen here, kept only as a defensive fallback).
    """
    G = nx.DiGraph()
    G.add_nodes_from(columns)
    G.add_edges_from(_clean_edges(true_edges))
    try:
        generations = list(nx.topological_generations(G))
    except nx.NetworkXUnfeasible:
        return nx.spring_layout(G, seed=42)
    pos = {}
    for depth, layer in enumerate(generations):
        ordered = sorted(layer)
        for i, node in enumerate(ordered):
            pos[node] = (depth, -(i - (len(ordered) - 1) / 2))
    return pos


def _draw_comparison(ax, true_edges, pred_edges, columns, pos, title):
    """
    Draws one method's true-vs-predicted comparison as a single overlaid
    graph: correctly recovered edges (TP), spurious predicted edges (FP),
    and missed true edges (FN) are each drawn in a distinct color/style --
    the standard single-figure comparison convention used in
    causal-discovery papers (e.g. NOTEARS, DAG-GNN), more information-dense
    than two separate side-by-side panels.
    """
    G = nx.DiGraph()
    G.add_nodes_from(columns)
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color='#EAEAEA', edgecolors='black', node_size=450)
    nx.draw_networkx_labels(G, pos, ax=ax, font_size=8)

    if pred_edges is None:
        ax.set_title(f"{title}\n(no result)", fontsize=9, color='#999999')
        ax.axis('off')
        return

    true_set = _clean_edges(true_edges)
    pred_set = _clean_edges(pred_edges)
    tp = true_set & pred_set
    fp = pred_set - true_set
    fn = true_set - pred_set

    def draw(edges, color, style, width):
        if edges:
            nx.draw_networkx_edges(G, pos, ax=ax, edgelist=list(edges), edge_color=color,
                                    style=style, width=width, arrowsize=9,
                                    connectionstyle='arc3,rad=0.06')

    draw(fn, '#999999', 'dotted', 1.1)   # true edge the method missed
    draw(fp, '#C0392B', 'dashed', 1.1)   # spurious edge the method invented
    draw(tp, '#2E7D32', 'solid', 1.7)    # correctly recovered edge

    ax.set_title(title, fontsize=9)
    ax.axis('off')


# All 10 methods, same order as the summary table above.
PLOT_ORDER = ['PC', 'FCI', 'CD-NOD', 'LiNGAM', 'DAG-GNN', 'GES',
              'PCMCI', 'PCMCI+', 'TCDF', 'LPCMCI']
IID_METHODS = {'PC', 'FCI', 'CD-NOD', 'LiNGAM', 'DAG-GNN', 'GES'}

iid_pos = _layered_layout(iid_truth, iid_cols)
ts_pos = _layered_layout(ts_truth, ts_cols)

fig, axes = plt.subplots(2, 5, figsize=(24, 10))
for ax, name in zip(axes.flatten(), PLOT_ORDER):
    if name in IID_METHODS:
        true_e, cols, pos, family = iid_truth, iid_cols, iid_pos, 'IID'
    else:
        true_e, cols, pos, family = ts_truth, ts_cols, ts_pos, 'TimeSeries'
    subtitle = f"{name}  ({family})"
    if name in PAG_METHODS:
        subtitle += "\n[PAG: directed output shown; headline score is skeleton-based]"
    _draw_comparison(ax, true_e, predicted_edges.get(name), cols, pos, subtitle)

legend_elems = [
    Line2D([0], [0], color='#2E7D32', lw=1.7, label='Correct (true positive)'),
    Line2D([0], [0], color='#C0392B', lw=1.1, linestyle='dashed', label='Spurious (false positive)'),
    Line2D([0], [0], color='#999999', lw=1.1, linestyle='dotted', label='Missed (false negative)'),
]
fig.legend(handles=legend_elems, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.04), fontsize=11, frameon=False)
fig.suptitle('True vs. Predicted Causal Graphs — All Methods', y=1.08, fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('true_vs_predicted_graphs_bench.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved -> true_vs_predicted_graphs_bench.png")
